# AgriLLM Inference Server (Colab)

Serves `unsloth/Phi-4` (~14B, 4-bit) behind a FastAPI `/generate` endpoint, exposed via an ngrok tunnel.

> **Note:** The `DARJYO/darjyo-AgriLLM-GRPO` repo on Hugging Face is a broken upload — it contains no model weights or tokenizer, only a training script. So we load `unsloth/Phi-4` (the exact base model that script fine-tunes from) and rely on the backend's agriculture prompt. For the genuine AgriLLM fine-tune, see the AI71 GGUF note at the bottom.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

**Steps:** run each cell top to bottom. The last cell prints a public URL like `https://xxxx.ngrok-free.app`. Paste it into your backend's `.env` as `AGRILLM_URL`.

In [ ]:
# 1. Install dependencies
!pip -q install unsloth fastapi uvicorn pyngrok nest-asyncio

In [ ]:
# 2. (Optional) Hugging Face login if the model is gated.
# Get a token at https://huggingface.co/settings/tokens
from huggingface_hub import login
# login(token="hf_xxx")  # uncomment and paste your token if download fails

In [ ]:
# 3. Load the model in 4-bit (via unsloth)
import torch
from unsloth import FastLanguageModel

MODEL_ID = "unsloth/Phi-4"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)  # enable fast inference path
print("Model loaded.")

In [ ]:
# 4. Generation helper
def generate_text(prompt, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.4,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print(generate_text("Say hello in one short sentence."))

In [ ]:
# 5. FastAPI app exposing /generate and /health
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class GenRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 512

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/generate")
def generate(req: GenRequest):
    text = generate_text(req.prompt, req.max_new_tokens)
    return {"text": text}

In [ ]:
# 6. Start the tunnel + server
# Get a free authtoken at https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok
import threading, time, uvicorn

# Close any old tunnels (free tier allows only 1).
ngrok.kill()

ngrok.set_auth_token("PASTE_YOUR_NGROK_AUTHTOKEN_HERE")

# Run uvicorn in a background thread so it doesn't clash with Colab's event loop.
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

# Wait until the server is actually listening before opening the tunnel.
time.sleep(5)

public_url = ngrok.connect(8000)
print("PUBLIC URL (set as AGRILLM_URL in your backend .env):", public_url)
print("Leave this cell running. Test it at:", f"{public_url}/health")


## Optional: use the genuine AgriLLM fine-tune

`unsloth/Phi-4` above is the base model. If you want the real **AI71 AgriLLM** fine-tune, it's only
practical as a quantized GGUF. On a free T4 (16 GB) the `Q3_K_S` quant (~13 GB) is the largest that fits:

```python
!pip -q install llama-cpp-python huggingface_hub
from huggingface_hub import hf_hub_download
gguf = hf_hub_download(
    repo_id="mradermacher/agrillm-Qwen3-30B-A3B-GGUF",
    filename="agrillm-Qwen3-30B-A3B.Q3_K_S.gguf",
)
from llama_cpp import Llama
llm = Llama(model_path=gguf, n_gpu_layers=-1, n_ctx=2048)
# Then change generate_text() to call: llm(prompt, max_tokens=512)["choices"][0]["text"]
```

This is slower to download and tighter on memory, so use it only if you specifically need the AI71 fine-tune.
